In [1]:
# 0. imports, SID4 parameters, and seeds
import os
import random

import numpy as np
import pandas as pd

from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import PromptTemplate
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

SID4 = 670
SEED = SID4
SLICE = SID4 % 1000
HP_ID = SID4 % 6
CLS_A = SID4 % 10
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10

random.seed(SEED)
np.random.seed(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("SID4: %04d" % SID4)
print("SEED:", SEED)
print("SLICE:", SLICE)
print("HP_ID:", HP_ID)
print("CLS_A:", CLS_A)
print("CLS_B:", CLS_B)
print("HP_ID is reported only. HW2 has no HP_ID mapping.")


/var/folders/g4/nd6wfspd3sx8dm7ps7vg9dn80000gn/T/ipykernel_58308/3416260951.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WikipediaLoader
/Users/snehasingh/Desktop/Gen-Ai/.venv-hw1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SID4: 0670
SEED: 670
SLICE: 670
HP_ID: 4
CLS_A: 0
CLS_B: 5
HP_ID is reported only. HW2 has no HP_ID mapping.


# Sneha Singh
## DATA 266 — Homework 2, Part 2
RAG on 10 Wikipedia movie pages with LangChain


## Part 2. RAG pipeline (4 points)

I load 10 Wikipedia movie pages with a LangChain loader, split them (chunk 500, overlap 50), embed the chunks, and store them in a vector store.

The pipeline is separate pieces: PromptTemplate, retriever, LLM. I do not use one pre-built chain.

I picked some titles that collide (`The Batman` / `Batman Begins` / `The Dark Knight`, and two *Dune* films) so retrieval can fail. Unique pages are Inception, Interstellar, Parasite, Titanic, The Godfather.

LLM is local Flan-T5-small (no API key). Embeddings are `all-MiniLM-L6-v2`.


In [2]:
# 2.1a Wikipedia page titles and the 5 questions
MOVIES = [
    "Inception",
    "Interstellar (film)",
    "The Dark Knight",
    "Batman Begins",
    "The Batman (film)",
    "Dune (2021 film)",
    "Dune: Part Two",
    "Parasite (2019 film)",
    "Titanic (1997 film)",
    "The Godfather",
]

QUESTIONS = [
    "Who directed The Batman?",
    "In what year was Dune released as a film?",
    "Which award did Parasite win at the Cannes Film Festival?",
    "Who composed the score for Inception?",
    "Who played Jack Dawson in Titanic (1997)?",
]


### 2.1 Load 10 documents with a LangChain Document Loader


In [3]:
# 2.1b load one Wikipedia page per movie title
docs = []
for title in MOVIES:
    loaded = WikipediaLoader(
        query=title,
        load_max_docs=1,
        doc_content_chars_max=15000,
    ).load()
    for d in loaded:
        d.metadata["movie"] = title
    docs.extend(loaded)
    print(title, "->", loaded[0].metadata.get("title"), "chars:", len(loaded[0].page_content))

print("documents:", len(docs))


Inception -> Inception chars: 15000
Interstellar (film) -> Interstellar (film) chars: 15000
The Dark Knight -> The Dark Knight chars: 15000
Batman Begins -> Batman Begins chars: 15000
The Batman (film) -> The Batman (film) chars: 15000
Dune (2021 film) -> Dune (2021 film) chars: 15000
Dune: Part Two -> Dune: Part Two chars: 15000
Parasite (2019 film) -> Parasite (2019 film) chars: 15000
Titanic (1997 film) -> Titanic (1997 film) chars: 15000
The Godfather -> The Godfather chars: 15000
documents: 10


### 2.2 Split into chunks (size 500, overlap 50)


In [4]:
# 2.2 split pages; 500 / 50 is the assignment default
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)
print("chunks:", len(chunks))
print("example metadata:", chunks[0].metadata)
print(chunks[0].page_content[:300])


chunks: 439
example metadata: {'title': 'Inception', 'summary': 'Inception is a 2010  science fiction action film written and directed by Christopher Nolan, who also produced it with Emma Thomas, his wife. The film stars Leonardo DiCaprio as a professional thief who steals information by infiltrating the subconscious of his targets. He is offered a chance to have his criminal history erased as payment for the implantation of another person\'s idea into a target\'s subconscious. The ensemble cast includes Ken Watanabe, Joseph Gordon-Levitt, Marion Cotillard, Elliot Page, Tom Hardy, Cillian Murphy, Tom Berenger, Dileep Rao, and Michael Caine.\nAfter the completion of Insomnia in 2002, Nolan presented to Warner Bros. a written 80-page treatment for a horror film envisioning "dream stealers," based on lucid dreaming. Deciding he needed more experience before tackling a production of this magnitude and complexity, Nolan shelved the project and instead worked on Batman Begins (2005), The Pre

### 2.3 Embed chunks and store in a vector store


In [5]:
# 2.3 MiniLM embeddings + in-memory vector store
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
store = InMemoryVectorStore.from_documents(chunks, embeddings)
retriever = store.as_retriever(search_kwargs={"k": 3})
print("vector store ready, k=3")


/var/folders/g4/nd6wfspd3sx8dm7ps7vg9dn80000gn/T/ipykernel_58308/434917088.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 5587.78it/s]


vector store ready, k=3


### 2.4 Prompt template, retriever, and LLM (not one pre-built chain)


In [6]:
# 2.4 prompt + local Flan-T5; retrieve and generate are called separately below
# transformers 5 has no text2text-generation task, so I call generate() myself
prompt = PromptTemplate.from_template(
    "Answer using only the context. If the context is missing the answer, say you do not know.\n\n"
    "Context:\n{context}\n\nQuestion: {question}\nAnswer:"
)

tok = AutoTokenizer.from_pretrained("google/flan-t5-small")
t5 = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")


class FlanLLM:
    def invoke(self, text):
        inputs = tok(text, return_tensors="pt", truncation=True, max_length=512)
        out = t5.generate(**inputs, max_new_tokens=64)
        return tok.decode(out[0], skip_special_tokens=True)


llm = FlanLLM()


def run_rag(question, retriever, prompt, llm, k=3):
    # retriever first, then fill the prompt, then LLM — not RetrievalQA
    hits = retriever.invoke(question)
    context = "\n\n".join(h.page_content for h in hits)
    text = prompt.format(context=context, question=question)
    answer = llm.invoke(text)
    return hits, answer


Loading weights: 100%|█████████████████████| 190/190 [00:00<00:00, 9780.65it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


### 2.5 Five questions: retrieved chunks and generated answers


In [7]:
# 2.5 run each question through retriever + LLM (chunk 500 / overlap 50)
results_500 = []
for q in QUESTIONS:
    hits, answer = run_rag(q, retriever, prompt, llm)
    results_500.append((q, hits, answer))
    print("=" * 80)
    print("Q:", q)
    print("A:", answer)
    for i, h in enumerate(hits, start=1):
        src = h.metadata.get("movie") or h.metadata.get("title")
        print(f"\n--- retrieved {i} | {src} ---")
        print(h.page_content)


Q: Who directed The Batman?
A: Matt Reeves

--- retrieved 1 | The Batman (film) ---
The Batman is a 2022 American superhero film directed by Matt Reeves from a screenplay he wrote with Peter Craig. Based on the DC Comics character Batman, it is a reboot of the Batman film franchise. Robert Pattinson stars as Bruce Wayne / Batman alongside Zoë Kravitz, Paul Dano, Jeffrey Wright, John Turturro, Peter Sarsgaard, Andy Serkis, and Colin Farrell. The film sees Batman, in his second year fighting crime in Gotham City, uncover corruption with ties to his own family while pursuing the

--- retrieved 2 | The Batman (film) ---
== Production ==

After Ben Affleck was cast as Bruce Wayne / Batman for the DC Extended Universe (DCEU) franchise in 2013, he began developing a standalone Batman film for him to star in, which Warner Bros. Pictures announced in October 2014. Affleck was also attached to direct and co-write the script with Geoff Johns b

--- retrieved 3 | The Batman (film) ---
After facing

### 2.6 Change chunk size for 2 questions and compare

I re-run **Q1** (The Batman) and **Q2** (Dune year) with chunk 200, overlap 20. Smaller chunks can retrieve a tighter sentence but can also cut the director / year off the chunk.


In [8]:
# 2.6 rebuild store with chunk 200 / overlap 20, re-run Q1 and Q2
splitter_small = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks_small = splitter_small.split_documents(docs)
store_small = InMemoryVectorStore.from_documents(chunks_small, embeddings)
retriever_small = store_small.as_retriever(search_kwargs={"k": 3})
print("small chunks:", len(chunks_small))

for q in QUESTIONS[:2]:
    hits, answer = run_rag(q, retriever_small, prompt, llm)
    print("=" * 80)
    print("Q (chunk 200/20):", q)
    print("A:", answer)
    for i, h in enumerate(hits, start=1):
        src = h.metadata.get("movie") or h.metadata.get("title")
        print(f"\n--- retrieved {i} | {src} ---")
        print(h.page_content)


small chunks: 970
Q (chunk 200/20): Who directed The Batman?
A: Matt Reeves

--- retrieved 1 | The Batman (film) ---
The Batman is a 2022 American superhero film directed by Matt Reeves from a screenplay he wrote with Peter Craig. Based on the DC Comics character Batman, it is a reboot of the Batman film franchise.

--- retrieved 2 | Batman Begins ---
Batman (1989). Receiving a nomination for the Academy Award for Best Cinematography, the film elevated Bale to leading man status while it made Nolan a high-profile director.

--- retrieved 3 | Batman Begins ---
the role to Anthony Hopkins but he declined. Nolan went to Caine's country home to personally deliver him the script, telling what his role would be and describing Alfred as "Batman's godfather".
Q (chunk 200/20): In what year was Dune released as a film?
A: 2021

--- retrieved 1 | Dune: Part Two ---
Legendary released Dune in the US on October 22, 2021 through a distribution pact with Warner Bros. Pictures. The film saw immediate

### 2.7 Retrieval scoring (top-3)

I scored the **500 / 50** run by hand. Success = the gold fact is in at least one of the top-3 chunks. The LLM answer can still be wrong.

Gold:

1. The Batman (2022) — **Matt Reeves**
2. Dune year in this corpus — **2021** (Villeneuve Part One). Part Two is 2024
3. Parasite at Cannes — **Palme d'Or**
4. Inception score — **Hans Zimmer**
5. Jack Dawson — **Leonardo DiCaprio**

| Q | Gold passage | In top-3? | Rank of first relevant | Notes |
|---|--------------|-----------|------------------------|-------|
| 1 | "directed by Matt Reeves" on The Batman (film) | Yes | 1 | Rank 1 is the right page |
| 2 | "is a 2021" / US release 2021 | Yes | 1 | Rank 1 is *Dune: Part Two* talking about the 2021 film, rank 2 is the 2021 page |
| 3 | Palme d'Or (same chunk also names Best Picture) | Yes | 1 | Palme d'Or is in rank 1, but the LLM still said Oscar |
| 4 | Hans Zimmer composed Inception | **No** | — | Top-3 are Nolan / screenplay / Oscars. Zimmer never shows up |
| 5 | "Leonardo DiCaprio as Jack Dawson" | Yes | 2 | Rank 1 is the Halifax grave labeled J. Dawson |

Retrieval Success Rate = 4 / 5 = **0.80**


In [10]:
# 2.7 scoring table for chunk 500 / overlap 50 (from the chunks I printed above)
score_df = pd.DataFrame([
    {"Q": 1, "gold": "Matt Reeves directed The Batman (film)", "in_top3": "Yes", "rank_first_relevant": "1", "notes": "right page at rank 1"},
    {"Q": 2, "gold": "Dune (2021 film) year 2021", "in_top3": "Yes", "rank_first_relevant": "1", "notes": "year is there; rank 1 is Part Two page"},
    {"Q": 3, "gold": "Parasite won Palme d'Or at Cannes", "in_top3": "Yes", "rank_first_relevant": "1", "notes": "Palme d'Or in rank 1; LLM still said Oscar"},
    {"Q": 4, "gold": "Hans Zimmer composed Inception", "in_top3": "No", "rank_first_relevant": "—", "notes": "composer not in top-3"},
    {"Q": 5, "gold": "Leonardo DiCaprio played Jack Dawson", "in_top3": "Yes", "rank_first_relevant": "2", "notes": "rank 1 is the J. Dawson grave"},
])
print(score_df.to_string(index=False))
print("Retrieval Success Rate = 4 / 5 = 0.80")
score_df


 Q                                   gold in_top3 rank_first_relevant                                      notes
 1 Matt Reeves directed The Batman (film)     Yes                   1                       right page at rank 1
 2             Dune (2021 film) year 2021     Yes                   1     year is there; rank 1 is Part Two page
 3      Parasite won Palme d'Or at Cannes     Yes                   1 Palme d'Or in rank 1; LLM still said Oscar
 4         Hans Zimmer composed Inception      No                   —                      composer not in top-3
 5   Leonardo DiCaprio played Jack Dawson     Yes                   2              rank 1 is the J. Dawson grave
Retrieval Success Rate = 4 / 5 = 0.80


,Q,gold,in_top3,rank_first_relevant,notes
0,1,Matt Reeves directed The Batman (film),Yes,1,right page at rank 1
1,2,Dune (2021 film) year 2021,Yes,1,year is there; rank 1 is Part Two page
2,3,Parasite won Palme d'Or at Cannes,Yes,1,Palme d'Or in rank 1; LLM still said Oscar
3,4,Hans Zimmer composed Inception,No,—,composer not in top-3
4,5,Leonardo DiCaprio played Jack Dawson,Yes,2,rank 1 is the J. Dawson grave


### 2.8 At least two RAG failures

**Failure A — Q4: correct document not retrieved / relevant chunk ranked too low**

Question: who composed the score for Inception? Gold is Hans Zimmer.

The retriever did get the Inception page, but the top-3 chunks are the opening (Nolan, DiCaprio), a screenplay paragraph, and the box-office / Oscar list. Chunk 3 says the film was nominated for Best Original Score and never names Zimmer. I loaded only the first 15,000 characters of each Wikipedia page, so the music / composer section may not even be in the document. The LLM then answered **Nolan**, which is in chunk 1. So this is retrieval first: the composer string was not in the top-3 (and maybe not in the truncated page). Then the LLM filled the gap with the director.

**Failure B — Q3: correct context retrieved but LLM answered incorrectly**

Question: which award did Parasite win at Cannes? Gold is Palme d'Or.

Rank 1 is the right page and it does say Palme d'Or. The same chunk also says Parasite won the Academy Award for Best Picture, and that sentence comes first. Flan-T5 answered **Academy Award for Best Picture**. The retriever did its job. The LLM ignored "Cannes" and copied the Oscar, which is unsupported as a Cannes award. Small local models do this when two awards sit in one chunk.

Same run, related to the chunk-size change:

- **Q2, ambiguous titles.** "What year was Dune released" pulled *Dune: Part Two* at rank 1. That page talks about the 2021 release of the first film, so 2021 is still in the chunk and I marked retrieval Yes. The ranking is still confused: Part Two outranked the 2021 film page.
- **Q1 with chunk 200/20.** Rank 1 stayed Matt Reeves / *The Batman*, so the answer stayed right. Rank 2 and 3 switched to *Batman Begins* (Nolan, Bale). Smaller chunks made the Batman name collision worse. Rank 1 of the 500/50 run was cleaner (all three hits were *The Batman*).
